# 04 — Modeling

Trains a classifier to predict KOI disposition (CONFIRMED / CANDIDATE /
FALSE POSITIVE) from the cleaned physical features.

Approach: split data into train/test so we can measure real learning (not
memorization), build a simple **baseline first** (logistic regression) to set
a number to beat, then try a stronger model. Because the classes are imbalanced
(~51% false positives), we judge models on **precision/recall/F1**, not accuracy.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

In [2]:
df = pd.read_csv("../data/processed/koi_cleaned.csv")
print("Shape:", df.shape)
print("Target present:", "koi_disposition" in df.columns)
df.head()

Shape: (9564, 106)
Target present: True


,koi_gmag,koi_rmag,koi_imag,koi_zmag,koi_jmag,koi_jmag_err,koi_hmag,koi_hmag_err,koi_kmag,koi_kmag_err,...,koi_time0,koi_time0_err1,koi_time0_err2,koi_insol,koi_insol_err1,koi_insol_err2,koi_srho,koi_srho_err1,koi_srho_err2,koi_fittype
0,15.890,15.270,15.114,15.006,14.082,0.025,13.751,0.030,13.648,0.054,...,2455003.539,0.002160,-0.002160,93.59,29.45,-16.65,3.20796,0.33173,-1.09986,LS+MCMC
1,15.890,15.270,15.114,15.006,14.082,0.025,13.751,0.030,13.648,0.054,...,2454995.514,0.003520,-0.003520,9.11,2.87,-1.62,3.02368,2.20489,-2.49638,LS+MCMC
2,15.943,15.390,15.220,15.166,14.254,0.028,13.900,0.033,13.826,0.058,...,2455008.850,0.000581,-0.000581,39.30,31.04,-10.49,7.29555,35.03293,-2.75453,LS+MCMC
3,16.100,15.554,15.382,15.266,14.326,0.035,13.911,0.042,13.809,0.048,...,2455003.308,0.000115,-0.000115,891.96,668.95,-230.35,0.22080,0.00917,-0.01837,LS+MCMC
4,16.015,15.468,15.292,15.241,14.366,0.033,14.064,0.047,13.952,0.047,...,2455004.596,0.001130,-0.001130,926.16,874.33,-314.24,1.98635,2.71141,-1.74541,LS+MCMC


In [3]:
# X = features (everything the model learns FROM)
# y = target (the label the model predicts)
X = df.drop(columns=["koi_disposition"])    # drop the target to get features
y = df["koi_disposition"]                    # the target column

print("Features:", X.shape)
print("Target distribution:\n", y.value_counts())

Features: (9564, 105)
Target distribution:
 koi_disposition
FALSE POSITIVE    4839
CONFIRMED         2747
CANDIDATE         1978
Name: count, dtype: int64


In [4]:
# Which columns are still text (object/string), not numbers?
X.select_dtypes(exclude="number").columns.tolist()

['koi_delivname',
 'koi_vet_stat',
 'koi_vet_date',
 'koi_disp_prov',
 'koi_parm_prov',
 'koi_fittype']

In [5]:
# Leftover text metadata: delivery names, vetting status/date, provenance
# labels, fit type. Bookkeeping about how the data was produced, not physical
# properties — same category as the metadata dropped in notebook 03. Some
# (e.g. disp_prov) are also mildly leaky since they describe how the label was set.
text_cols = X.select_dtypes(exclude="number").columns.tolist()
X = X.drop(columns=text_cols)
print("Dropped:", text_cols)
print("Features now:", X.shape)

Dropped: ['koi_delivname', 'koi_vet_stat', 'koi_vet_date', 'koi_disp_prov', 'koi_parm_prov', 'koi_fittype']
Features now: (9564, 99)


In [6]:
# Hold out 20% of the data as a test set the model NEVER sees during training.
# stratify=y keeps the same class proportions (~51/29/20) in both train and test.
# random_state=42 makes the split reproducible — same rows every run.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (7651, 99)
Test: (1913, 99)


In [7]:
# Baseline: logistic regression. Simple, fast, interpretable — the number to beat.
baseline = LogisticRegression(max_iter=1000)
baseline.fit(X_train, y_train)        # learn patterns from the training data ONLY

y_pred = baseline.predict(X_test)     # predict on the held-out test set
print("Done — model trained and predictions made.")

Done — model trained and predictions made.


/Users/husnainabbas/exoplanet-triage/venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [8]:
# Scale features to mean 0 / std 1 so no feature dominates by magnitude alone.
# Fit the scaler on TRAIN ONLY, then apply to both — the test set must stay
# "unseen," so we don't let it influence the scaling.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

baseline = LogisticRegression(max_iter=1000)
baseline.fit(X_train_scaled, y_train)
y_pred = baseline.predict(X_test_scaled)
print("Trained on scaled features.")

Trained on scaled features.


In [9]:
# Precision/recall/F1 per class — the honest metrics given the class imbalance.
print(classification_report(y_test, y_pred))

# Confusion matrix: rows = actual class, columns = predicted class.
print("Confusion matrix (rows=actual, cols=predicted):")
print(confusion_matrix(y_test, y_pred))

                precision    recall  f1-score   support

     CANDIDATE       0.62      0.50      0.56       396
     CONFIRMED       0.82      0.89      0.85       549
FALSE POSITIVE       0.86      0.89      0.88       968

      accuracy                           0.81      1913
     macro avg       0.77      0.76      0.76      1913
  weighted avg       0.80      0.81      0.80      1913

Confusion matrix (rows=actual, cols=predicted):
[[199  78 119]
 [ 43 487  19]
 [ 78  27 863]]


In [10]:
# Random Forest: ~hundreds of decision trees, each trained on random subsets
# of rows and features, then majority-voting. Diversity cancels out individual
# trees' overfitting. Note: trees use raw feature values (no scaling needed) —
# so we feed UNSCALED X_train/X_test, unlike logistic regression.
rf = RandomForestClassifier(
    n_estimators=300,      # number of trees in the forest
    random_state=42,       # reproducible (same forest every run)
    n_jobs=-1,             # use all CPU cores — trees train in parallel
)
rf.fit(X_train, y_train)          # note: UNSCALED
rf_pred = rf.predict(X_test)      # note: UNSCALED

print("Random Forest — results:")
print(classification_report(y_test, rf_pred))
print("Confusion matrix (rows=actual, cols=predicted):")
print(confusion_matrix(y_test, rf_pred))

Random Forest — results:
                precision    recall  f1-score   support

     CANDIDATE       0.72      0.62      0.67       396
     CONFIRMED       0.91      0.90      0.90       549
FALSE POSITIVE       0.88      0.94      0.91       968

      accuracy                           0.86      1913
     macro avg       0.84      0.82      0.83      1913
  weighted avg       0.86      0.86      0.86      1913

Confusion matrix (rows=actual, cols=predicted):
[[246  43 107]
 [ 43 493  13]
 [ 52   6 910]]


In [11]:
# Which features did the forest rely on most? Sanity check + preview of nb 05.
importances = pd.Series(rf.feature_importances_, index=X.columns)
importances.sort_values(ascending=False).head(15)

koi_dicco_msky       0.044987
koi_dikco_msky       0.044211
koi_prad             0.037212
koi_smet_err2        0.032026
koi_ror              0.026553
koi_fwm_stat_sig     0.024549
koi_model_snr        0.023907
koi_max_mult_ev      0.023706
koi_steff_err2       0.021205
koi_smet_err1        0.021166
koi_steff_err1       0.020563
koi_dor              0.020037
koi_prad_err1        0.018519
koi_count            0.017921
koi_duration_err2    0.017612
dtype: float64

## Feature Importance

The most predictive features are physically sensible, which is evidence the
model learned real astronomy rather than noise:
- **`koi_prad`** (planet radius) ranks high — matching the notebook-02 finding
  that radius separates confirmed planets (~2.9 Earth radii) from false
  positives (~165, often eclipsing binaries).
- **`koi_ror`, `koi_dor`** (transit geometry ratios), **`koi_model_snr`**
  (signal-to-noise), and **stellar temperature** terms also rank highly — all
  real, physically meaningful measurements.

**A leakage note worth flagging:** the top two features, `koi_dicco_msky` and
`koi_dikco_msky`, are centroid-offset measurements — closely related to the
`koi_fpflag_co` (centroid-offset false-positive flag) that was dropped as
leakage in notebook 03. These are raw measurements rather than verdicts, so
they were kept as legitimate features. But they sit near the leakage boundary:
the model may be partly re-deriving NASA's own vetting logic rather than
finding fully independent signal. A stricter version of this project could drop
them to test how the model performs on purely independent features. Importance
is spread across many features (top feature only ~4.5%), so no single
answer-key column dominates.

In [12]:
# XGBoost requires numeric class labels, not strings. LabelEncoder maps
# CANDIDATE/CONFIRMED/FALSE POSITIVE -> 0/1/2 (and back, for readable reports).
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)   # learn mapping on train, encode it
y_test_enc = le.transform(y_test)         # apply same mapping to test

print("Classes:", list(le.classes_))      # shows which number = which class

Classes: ['CANDIDATE', 'CONFIRMED', 'FALSE POSITIVE']


In [13]:
# XGBoost: gradient-boosted trees. Builds trees sequentially, each correcting
# the previous ones' errors (vs RF's independent-trees-then-vote). Usually the
# strongest model for tabular data. Trees -> no scaling needed (unscaled X).
xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.1,     # how much each new tree corrects — smaller = more careful
    max_depth=6,           # how deep each tree can go
    random_state=42,
    n_jobs=-1,
)
xgb.fit(X_train, y_train_enc)
xgb_pred = xgb.predict(X_test)

# Decode numeric predictions back to class names for a readable report
xgb_pred_labels = le.inverse_transform(xgb_pred)

print("XGBoost — results:")
print(classification_report(y_test, xgb_pred_labels))
print("Confusion matrix (rows=actual, cols=predicted):")
print(confusion_matrix(y_test, xgb_pred_labels))

XGBoost — results:
                precision    recall  f1-score   support

     CANDIDATE       0.70      0.64      0.66       396
     CONFIRMED       0.90      0.91      0.90       549
FALSE POSITIVE       0.90      0.92      0.91       968

      accuracy                           0.86      1913
     macro avg       0.83      0.82      0.83      1913
  weighted avg       0.86      0.86      0.86      1913

Confusion matrix (rows=actual, cols=predicted):
[[252  49  95]
 [ 44 498   7]
 [ 66   8 894]]


In [14]:
# Build a small, recognizable sample for the live demo dropdown.
# kepoi_name was dropped in nb 03 as a non-predictive identifier — the model
# never saw it. The demo needs something human-readable to show, so we bring
# it back here for display only, joined by row index (preserved end-to-end
# since no rows were ever dropped, only columns).
raw = pd.read_csv("../data/raw/koi_cumulative.csv")
kepoi_names = raw.loc[X_test.index, "kepoi_name"]

# A handful of real test-set rows per class, so the dropdown shows variety
# instead of 20 near-identical CONFIRMED entries.
demo_rows = []
for cls in y_test.unique():
    cls_idx = y_test[y_test == cls].index[:7]
    demo_rows.extend(cls_idx)

demo_features = X_test.loc[demo_rows].reset_index(drop=True)
demo_labels = y_test.loc[demo_rows].reset_index(drop=True)
demo_names = kepoi_names.loc[demo_rows].reset_index(drop=True)

demo_sample = demo_features.copy()
demo_sample.insert(0, "kepoi_name", demo_names)
demo_sample.insert(1, "true_label", demo_labels)

print("Demo sample shape:", demo_sample.shape)
demo_sample[["kepoi_name", "true_label"]]

Demo sample shape: (21, 101)


,kepoi_name,true_label
0,K03593.01,FALSE POSITIVE
1,K03637.01,FALSE POSITIVE
2,K08281.01,FALSE POSITIVE
3,K00045.01,FALSE POSITIVE
4,K00231.01,FALSE POSITIVE
5,K08237.01,FALSE POSITIVE
6,K01810.01,FALSE POSITIVE
7,K01339.01,CONFIRMED
8,K02890.01,CONFIRMED
9,K00119.01,CONFIRMED


In [15]:
import joblib
from pathlib import Path

models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

# Everything the Streamlit app needs, bundled into one file:
# - the fitted model
# - the label encoder (turns 0/1/2 back into class names)
# - the exact feature column order the model expects
# - a small set of REAL, already-cleaned KOI rows for the dropdown
bundle = {
    "model": xgb,
    "label_encoder": le,
    "feature_columns": X.columns.tolist(),
    "demo_sample": demo_sample,
}

joblib.dump(bundle, models_dir / "exoplanet_model_bundle.joblib")
print("Saved bundle to", models_dir / "exoplanet_model_bundle.joblib")

Saved bundle to ../models/exoplanet_model_bundle.joblib


## Model Comparison

| Model | Macro F1 | Notes |
|---|---|---|
| Logistic Regression (baseline) | 0.76 | Linear boundaries; sets the bar to beat |
| Random Forest | 0.83 | Captures conditional feature interactions; large jump over baseline |
| XGBoost | 0.83 | Industry-standard boosting; ties RF, fewest dangerous errors |

The baseline-first approach pays off here: the 0.76 from logistic regression
makes the 0.83 from the tree models a *meaningful* number rather than one
floating in space — a 7-point macro-F1 gain from moving to models that handle
the conditional, non-linear patterns in the data ("large radius means false
positive, *unless* the period is also long...").

The most informative result is that **Random Forest and XGBoost — two strong
but mechanically different algorithms — land at the same 0.83.** When two good
models converge, it signals the limit is the *data's information content*, not
the choice of model. Pushing for a higher score with fancier modeling wouldn't
help, because the wall is what's knowable from these features, not the
algorithm.

That wall is the **CANDIDATE class** (F1 ~0.66 across all models). A candidate
is by definition a signal that hasn't been resolved into planet-or-not, so it
physically overlaps both other classes — even NASA's full pipeline leaves these
unresolved. The models reflect that ambiguity honestly rather than inventing
certainty that isn't in the data.

XGBoost is chosen as the final model: at the same macro F1, its confusion
matrix shows the fewest CONFIRMED→FALSE POSITIVE errors (7), meaning it most
rarely makes the costly mistake of discarding a real planet as junk.

## Summary

Built a three-model progression — logistic regression baseline (0.76) →
Random Forest (0.83) → XGBoost (0.83) — to classify KOIs as CONFIRMED,
CANDIDATE, or FALSE POSITIVE. Tree models substantially beat the linear
baseline; the two tree models tying suggests the data's information ceiling,
with the inherently-ambiguous CANDIDATE class as the limiting factor.

Evaluated on precision/recall/F1 (not accuracy) due to the ~51% false-positive
class imbalance. XGBoost chosen as final model for making the fewest costly
CONFIRMED→FALSE POSITIVE errors.

*Note:* a few leftover text/metadata columns (provenance, fit type, vetting
labels) were dropped here in notebook 04; ideally these belong in notebook 03's
cleaning step, and could be moved there in a future pass.

**Next:** notebook 05 — explainability (SHAP), to see *why* the model makes
individual predictions, supporting the triage use case.